---
title: Keecas Quarto Example
toc: true
format:
    html: default
    pdf: default
echo: false
---

# How to use keecas in a Quarto document

This example demonstrates all the key features of `keecas` for symbolic and units-aware calculations in a Quarto document.

In [1]:
# for quick start import *
# from keecas import *

# or explicit import (this is about all you need)
from keecas import symbols, u, pc, show_eqn, options, check

In [14]:
options.language = "it"

::: {.callout-note}

If the rendering engine is `KaTeX` (i.e. Jupyter notebook), the label command `\label{}` will result in a `ParseError`. The latex code will work when rendering by `quarto render`, but since you may want to see the result before rendering the document, an option to disable the `\label{}` command is provided.

When rendering with `quarto` you can have two options:

- if `qmd` files, then everything is fine
- if `ipynb` files, then pass the `--execute` flag to `quarto render` to rerender the notebook

:::


In [15]:
# | eval: false

# this option will not be set at rendering time (dev-mode)
options.katex = True  # disable `\label command`
options.PRINT_LABEL = (
    True  # print interpolated labels for easy reference (e.g. copy and paste)
)

In [16]:
# set a prefix for the latex equation
options.EQ_PREFIX = r"eq-QUARTO_EXAMPLE-"


# Initialize notebook-global dictionaries for persistence across cells
params = {}  # Global parameters that persist throughout the notebook
eqn = {}  # Global expressions that persist throughout the notebook

## Basic Symbolic Math with Units

Define symbols and calculate basic engineering quantities:

In [17]:
# Define symbols
F, A, sigma = symbols("F, A, sigma")

# Parameters with units
_p = {
    F: 10 * u.kN,
    A: 50 * u.cm**2,
}
params.update(_p)  # Save to global params

# Symbolic expressions
_e = {sigma: "F / A" | pc.parse_expr}
eqn.update(_e)  # Save to global expressions

# Evaluated results
_v = {k: v | pc.subs(_e | _p) | pc.convert_to([u.MPa]) | pc.N for k, v in _e.items()}

# Descriptions
_d = {F: "applied force", A: "cross-sectional area", sigma: "normal stress"}

# Labels
_l = {k: str(k) for k in _d.keys()}

show_eqn(
    [_p | _e, _v, _d],
    col_wrap=[None, "=", "=", (r"\quad(", ")")],
    float_format="{:.2f}",
    label=_l,
)

# the labels will be printed for easy reference, only in dev-mode (not rendered by quarto)

F: eq-QUARTO_EXAMPLE-F
A: eq-QUARTO_EXAMPLE-A
sigma: eq-QUARTO_EXAMPLE-sigma


\begin{align}
F & =\mathtt{\text{10.00 kilonewton}} &   & \quad(\mathtt{\text{applied force}})  \\[8pt]
 A & =\mathtt{\text{50.00 cm²}} &   & \quad(\mathtt{\text{cross-sectional area}})  \\[8pt]
 \sigma & =\dfrac{F}{A} & =2.00{\,}\text{MPa} & \quad(\mathtt{\text{normal stress}}) 
\end{align}

## Pipe Commands Demonstration {#sec-pipe}

Using pipe operators for functional composition:

In [18]:
# Chain operations using pipe commands
x, y, d = symbols("x, y, d")

# Parameters
_p = {
    x: 3 * u.m,
    y: 4 * u.m,
}

# Symbolic expressions
_e = {d: "x^2 + y^2" | pc.parse_expr}

# Evaluated results
_v = {k: v | pc.subs(_e | _p) | pc.convert_to([u.m]) | pc.N for k, v in _e.items()}

show_eqn([_p | _e, _v])

\begin{align}
x & =\mathtt{\text{3.00 m}} &    \\[8pt]
 y & =\mathtt{\text{4.00 m}} &    \\[8pt]
 d & =x^{2} + y^{2} & =25.0{\,}\text{m}^{2} 
\end{align}

## Different LaTeX Environments

### Cases Environment

In [19]:
# Material properties
E_steel, E_concrete = symbols("E_{steel}, E_{concrete}")

_p = {
    E_steel: 200 * u.GPa,
    E_concrete: 30 * u.GPa,
}

_d = {E_steel: "steel elastic modulus", E_concrete: "concrete elastic modulus"}

show_eqn([_p, _d], environment="cases", col_wrap=[None, "=", "&"])

\begin{align}
	\left\{\begin{aligned}
E_{steel} & =\mathtt{\text{200.00 gigapascal}} & &\mathtt{\text{steel elastic modulus}}  \\[8pt]
 E_{concrete} & =\mathtt{\text{30.00 gigapascal}} & &\mathtt{\text{concrete elastic modulus}} 	
\end{aligned}\right.
\end{align}

### Equation Environment with Labels {#sec-beam-calc}

Calculate beam deflection with cross-references:

In [20]:
# Beam calculation
q, L, E, I, delta = symbols("q, L, E, I, delta")

_p = {
    q: 5 * u.kN / u.m,
    L: 8 * u.m,
    E: 200 * u.GPa,
    I: 8360 * u.cm**4,
}
params.update(_p)  # Save to global params

# Deflection formula
_e = {delta: "5 * q * L^4 / (384 * E * I)" | pc.parse_expr}
eqn.update(_e)  # Save to global expressions

_v = {k: v | pc.subs(_e | _p) | pc.convert_to([u.mm]) | pc.N for k, v in _e.items()}

_l = {k: str(k) for k in _e.keys()}

show_eqn([_e, _v], label=_l, float_format="{:.3f}")

delta: eq-QUARTO_EXAMPLE-delta


\begin{align}
\delta & =\dfrac{5{\,}q{\,}L^{4}}{384{\,}E{\,}I} & =15.949{\,}\text{mm} 
\end{align}

::: {.callout-note}

labels emitted by `show_eqn` are latex labels, therefore they need to be referenced with `\eqref{}` or `\ref{}` latex commands.

:::

The deflection calculated in \ref{eq-QUARTO_EXAMPLE-delta} shows acceptable values for serviceability.

@eq-QUARTO_EXAMPLE-delta

## Verification Function

Using the `check` function for design checks:

In [21]:
# Design checks
sigma_Sd, tau_Sd, sigma_Rd, tau_Rd = symbols(
    r"\sigma_{Sd}, \tau_{Sd}, \sigma_{Rd}, \tau_{Rd}"
)

_p = {
    sigma_Sd: 20 * u.MPa,
    tau_Sd: 15 * u.MPa,
    sigma_Rd: 250 * u.MPa,
    tau_Rd: 75 * u.MPa,
}

# expressions to check
_expr = [
    sigma_Sd / sigma_Rd,
    tau_Sd / tau_Rd,
]

# evaluate expresions
_v = {k: k | pc.subs(_e | _p) | pc.N for k in _expr}

# check if expressions are less than 1
_c = {k: check(v, 1.0) for k, v in _v.items()}

# specify float format only for the check values
_ff = {k: [None, "{:.3f}", None] for k in _c.keys()}

show_eqn([_p | _v, _c], float_format=_ff)

\begin{align}
\sigma_{Sd} & =\mathtt{\text{20.00 MPa}} &    \\[8pt]
 \tau_{Sd} & =\mathtt{\text{15.00 MPa}} &    \\[8pt]
 \sigma_{Rd} & =\mathtt{\text{250.00 MPa}} &    \\[8pt]
 \tau_{Rd} & =\mathtt{\text{75.00 MPa}} &    \\[8pt]
 \dfrac{\sigma_{Sd}}{\sigma_{Rd}} & =0.080 & \qquad\textcolor{green}{\left[\le1.0\quad \textbf{VERIFICATO}\right]}  \\[8pt]
 \dfrac{\tau_{Sd}}{\tau_{Rd}} & =0.200 & \qquad\textcolor{green}{\left[\le1.0\quad \textbf{VERIFICATO}\right]} 
\end{align}

The type of comparison in the `check` function can be specified, as well the value to be specified against.

In [22]:
a, b, c, d, f = symbols("a, b, c, d, f")

from sympy import Le, Lt, Ge, Gt, Eq

_expr = {
    a: (3, Le, 1),
    b: (4, Gt, 2),
    c: (5, Lt, 3),
    d: (6, Ge, 4),
    f: (7, Eq, 5),
}

_c = {k: check(lhs=v[0], test=v[1], rhs=v[2]) for k, v in _expr.items()}

show_eqn(
    [
        {k: v[0] for k, v in _expr.items()},
        _c,
    ],
    col_wrap=[None, "=", "&"],
)

\begin{align}
a & =3 & &\textcolor{red}{\left[>1\quad \textbf{NON VERIFICATO}\right]}  \\[8pt]
 b & =4 & &\textcolor{green}{\left[>2\quad \textbf{VERIFICATO}\right]}  \\[8pt]
 c & =5 & &\textcolor{red}{\left[\ge3\quad \textbf{NON VERIFICATO}\right]}  \\[8pt]
 d & =6 & &\textcolor{green}{\left[\ge4\quad \textbf{VERIFICATO}\right]}  \\[8pt]
 f & =7 & &\textcolor{red}{\left[\neq5\quad \textbf{NON VERIFICATO}\right]} 
\end{align}

## Subs get ordered

Substitution will get ordered and sorted automatically, so you can use a symbol before mapping to a value.

In [23]:
# Complex expression with multiple substitutions
a, b, c, d = symbols("a, b, c, d")


_p = {
    a: 3,
    b: 4,
}
params.update(_p)

_e = {
    d: "sqrt(a^2 + b^2) / c" | pc.parse_expr,
    c: "a*b" | pc.parse_expr,
}
eqn.update(_e)

_v = {
    k: v | pc.subs(eqn|params) | pc.N for k, v in _e.items()
}

show_eqn([_p|_e, _v], float_format="{:.3f}")

\begin{align}
a & =3 &    \\[8pt]
 b & =4 &    \\[8pt]
 d & =\dfrac{\sqrt{a^{2} + b^{2}}}{c} & =0.417  \\[8pt]
 c & =a{\,}b & =12.000 
\end{align}

## Summary

This example demonstrated:

- Basic symbolic math with units
- Pipe command usage (see @sec-pipe)
- Different LaTeX environments (align, cases, equation)
- Label and cross-reference functionality
- Dataframe operations
- Verification functions
- Complex symbolic manipulations

All calculations in @sec-beam-calc show that keecas provides a powerful interface for engineering calculations in Quarto documents.